In [64]:
import json
import requests as rq
import pandas as pd
import pprint

In [24]:
with open("../_local/user_variables.json") as f:
    variables = json.load(f)

In [27]:
def get_token(client_id, client_secret, keycloak_url, username=None, password=None, access_token=False):
    payload = {
        "client_id": client_id,
        "client_secret": client_secret,
        "grant_type": "password",
        "username": username,
        "password": password,
        "scope": "openid",
    }
    response = rq.post(
        f"{keycloak_url}/auth/realms/candig/protocol/openid-connect/token",
        data=payload,
    )
    if response.status_code == 200:
        if access_token:
            return response.json()["access_token"]
        return response.json()["refresh_token"]

In [85]:
token = get_token(username=variables["SITE_ADMIN_USER"], password=variables["SITE_ADMIN_PASSWORD"], client_id=variables["CLIENT_ID"], client_secret=variables["CLIENT_SECRET"], keycloak_url=variables["KEYCLOAK_URL"])
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json; charset=utf-8"
}

In [72]:
federation_payload = {
    "method": "GET",
    "path": "query",
    "payload": {
        "primary_site": "Colon",
        "page_size":10000
    },
    "service": "query"
}

In [30]:
federation_url=f"{variables['CANDIG_URL']}/federation/v1/fanout"
print(federation_url)

http://10.11.0.5:5080/federation/v1/fanout


In [70]:
param={"page_size":10000}

In [35]:
print(federation_url)
print(federation_payload)

http://10.11.0.5:5080/federation/v1/fanout
{'method': 'GET', 'path': 'query', 'payload': {'treatment': 'Systemic therapy', 'primary_site': 'Colon'}, 'service': 'query'}


In [73]:
response = rq.post(federation_url, headers=headers, json=federation_payload)
pprint.pprint(response.json())

[{'location': {'name': 'LOCAL', 'province': 'ON', 'province-code': 'ca-on'},
  'results': {'count': 160,
              'genomic': [],
              'results': [{'cause_of_death': None,
                           'date_alive_after_lost_to_followup': None,
                           'date_of_birth': {'day_interval': -13094,
                                             'month_interval': -431},
                           'date_of_death': None,
                           'date_resolution': 'day',
                           'gender': 'Prefer not to disclose',
                           'is_deceased': None,
                           'lost_to_followup_after_clinical_event_identifier': None,
                           'lost_to_followup_reason': None,
                           'program_id': 'SiteA-SYNTH_03',
                           'sex_at_birth': 'Male',
                           'submitter_donor_id': 'SiteA-DONOR_0508'},
                          {'cause_of_death': 'Died of cancer',
    

In [74]:
len(response.json()[0]['results']['results'])

160

In [75]:
donors_by_location = {}
for location in response.json():
    print(location['location']['name'])
    programs = {}
    for donor in location['results']['results']:
        try:
            programs[donor['program_id']].append(donor['submitter_donor_id'])
        except KeyError as e:
            programs[donor['program_id']] = [donor['submitter_donor_id']]
pprint.pprint(programs)
          

LOCAL
{'SiteA-SYNTH_01': ['SiteA-DONOR_0143',
                    'SiteA-DONOR_0073',
                    'SiteA-DONOR_0118',
                    'SiteA-DONOR_0103',
                    'SiteA-DONOR_0113',
                    'SiteA-DONOR_0063',
                    'SiteA-DONOR_0093',
                    'SiteA-DONOR_0148',
                    'SiteA-DONOR_0028',
                    'SiteA-DONOR_0158',
                    'SiteA-DONOR_0153',
                    'SiteA-DONOR_0008',
                    'SiteA-DONOR_0138',
                    'SiteA-DONOR_0128',
                    'SiteA-DONOR_0098',
                    'SiteA-DONOR_0038',
                    'SiteA-DONOR_0108',
                    'SiteA-DONOR_0183',
                    'SiteA-DONOR_0003',
                    'SiteA-DONOR_0088',
                    'SiteA-DONOR_0048',
                    'SiteA-DONOR_0013',
                    'SiteA-DONOR_0163',
                    'SiteA-DONOR_0083',
                    'SiteA-DONOR_0

Get donor with clinical data for every listed donor

In [86]:
filtered_donors = []
for program in programs:
    for donor in programs[program]:
        donor_clinical_data_url=f"{variables['CANDIG_URL']}/katsu/v3/authorized/donor_with_clinical_data/program/{program}/donor/{donor}"
        response = rq.get(donor_clinical_data_url,
                         headers=headers)
        filtered_donors.append(response.json())

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200

In [87]:
filtered_donors[0]

{'submitter_donor_id': 'SiteA-DONOR_0508',
 'gender': 'Prefer not to disclose',
 'sex_at_birth': 'Male',
 'is_deceased': None,
 'lost_to_followup_after_clinical_event_identifier': None,
 'lost_to_followup_reason': None,
 'date_alive_after_lost_to_followup': None,
 'cause_of_death': None,
 'date_of_birth': {'day_interval': -13094, 'month_interval': -431},
 'date_of_death': None,
 'date_resolution': 'day',
 'program_id': 'SiteA-SYNTH_03',
 'primary_diagnoses': [{'submitter_primary_diagnosis_id': 'SiteA-DIAG_0508',
   'primary_site': 'Colon',
   'date_of_diagnosis': {'day_interval': 0, 'month_interval': 0},
   'cancer_type_code': 'C13',
   'basis_of_diagnosis': 'Histology of a metastasis',
   'laterality': 'Not a paired site',
   'clinical_tumour_staging_system': 'AJCC cancer staging system',
   'clinical_t_category': 'T2d',
   'clinical_n_category': 'N4',
   'clinical_m_category': 'M0',
   'clinical_stage_group': 'Stage IIIBES',
   'pathological_tumour_staging_system': 'AJCC cancer stagi

In [88]:
pd.DataFrame(filtered_donors[0])

ValueError: All arrays must be of the same length